# Nearest Neighbor

> 核心思想：训练阶段记住数据，测试阶段找最相似的训练样本。


## 1. 原理

### 1.1 训练和测试

**train:** memorize all data and labels  
**test:** predict the label of the most similar training image

也就是说，nearest neighbor 的训练过程几乎不学习参数，只是把训练集保存下来。真正的计算发生在测试阶段：对于一个新的样本 $x$，计算它和所有训练样本之间的距离，找到距离最近的训练样本，然后把这个训练样本的 label 作为预测结果。

可以把它理解成一种最简单的“局部近似”：如果两个点在特征空间里足够近，那么它们的输出也应该相似。


### 1.2 如何衡量 similarity?

$L_1$ distance:

$$
d(x,y) = \sum_i |x_i - y_i|
$$

![example](L1_distance.png)

$L_1$ distance 的直觉是逐维比较两个样本的差异，然后把所有差异加起来。对于图片来说，每一个 pixel 都可以看成一个维度，所以两张图片之间的距离就是所有 pixel 差异的总和。

但是这里有一个问题：pixel-level distance 不一定等价于 semantic similarity。比如一张图片整体平移几个像素，人看起来还是同一张图，但 $L_1$ distance 可能变化很大。


## 2. 拓展

### 2.1 训练快，推理慢

朴素 nearest neighbor 的推理复杂度很高，因为每次预测都要和训练集中所有样本计算距离。如果训练集很大，线上推理会很慢。

优化方向：[FAISS](https://github.com/facebookresearch/faiss) 这类工具解决的是 approximate nearest neighbor search，即用近似方法更快地找到足够近的样本。


### 2.2 From Nearest Neighbor to K-Nearest Neighbor

| K | 判断方式 |
|---:|---|
| 1 | 只看离它最近的 1 个样本，它是什么类，新样本就是什么类。 |
| 3 | 看最近的 3 个样本，哪个类别多，就判成哪个类别。 |
| 5 | 看最近的 5 个样本，以此类推。 |

$K$ 变大以后，模型会更平滑，不容易被单个噪声样本影响；但如果 $K$ 太大，又可能把局部结构抹掉，导致欠拟合。


### 2.3 From L1 Distance to Other Distance

$L_2$ distance:

$$
d(x,y) = \sqrt{\sum_i (x_i - y_i)^2}
$$

Cosine distance:

$$
d(x,y) = 1 - \frac{x \cdot y}{\|x\|\|y\|}
$$

question: 课程中说tf-idf是一种距离，但是就本人之前的经验而言，似乎他是一个embedding方法？

先用 tf-idf 把文本变成向量，再用 cosine similarity / cosine distance 衡量文本之间的相似性。


### 2.4 Pixel Distance 的问题

Distance metrics on pixels are not informative. 平移会带来非常大的影响。

**nearest neighbor with convnet:** 可以先用 CNN / ConvNet 把原始图片转成 feature embedding，再在 embedding space 中做 nearest neighbor。这样比较的就不是原始 pixel，而是神经网络学到的语义特征。


## 3. 超参

### 3.1 主要超参

| hyperparameter | candidates / notes |
|---|---|
| K | 1, 3, 5, 10, 20, 50, 100, 200, 500, 1000 |
| distance | L1, L2, cosine |
| preprocessing | normalization / standardization |

补充1：distance-based method 对尺度很敏感。如果某一维特征数值范围特别大，它会主导距离计算，所以是否做 normalization / standardization 也很重要。

### 3.2 L1 和 L2

L1：每个维度的差异是线性相加，对单个特别大的差异相对没那么敏感。  
L2：差异会平方后再相加，所以某个维度差很多时，影响会被放大。

L1 更适合高维、稀疏特征时试一试，比如文本、one-hot 特征。  
L2 更常用于连续数值特征，比如图像像素、坐标、传感器数据。

### 3.3 L1 和 L2 的几何直觉

L1 边界只有三种 orientation: horizontal, vertical, 45 degree 的，而 L2 边界可以任意 orientation。

![desicion boundary of 2 pixels' image](desicion_boundaries.png)

![L1, L2 distance](L1L2difference.png)


## 4. 调参

1. 三分法
2. cross validation (5 folds)

对于 KNN 这种离散超参，最常见的是 **grid search + cross validation**。比如枚举不同的 $K$ 和 distance metric，然后用 validation set 或 5-fold cross validation 选择平均表现最好的组合。

一般流程：

1. 把数据分成 train / validation / test，或者在 train 上做 cross validation。
2. 在 validation 或 cross validation 上选 $K$ 和 distance。
3. 最后只在 test set 上评估一次，避免把 test set 用成调参集。


## 5. 问题与理解

### 5.1 nn 是一种函数逼近器，按理来说可以模拟所有函数，但为何还要发明更多的算法呢？

因为“理论上能逼近”不等于“实际中好用”。nearest neighbor 的问题包括：推理慢、存储大、对距离度量敏感、对特征尺度敏感、在高维空间里需要非常多的数据才能覆盖空间。

其他算法通常会通过学习参数、提取特征、加入正则化等方式，提高泛化能力和推理效率。


### 5.2 一维函数到多维函数的转换

已有数据不可能覆盖维度中所有点，只能近似，而 nn 就是一种近似。

![dimensions](dimensions.png)

一维情况下，用一些采样点近似函数比较直观；但维度升高后，要保持相同的覆盖密度，需要的数据量会指数级增加。

因此 nearest neighbor 的核心假设是：在已有样本附近做局部近似。如果样本足够密、特征空间足够好，这个近似就有效；如果样本稀疏或特征空间不好，这个近似就会失效。



### 5.3 Curse of Dimensionality

![curse](curse_ofDimensionality.png)

维度越高，空间体积增长越快。有限数量的数据点会变得非常稀疏，因此“最近的点”也可能其实很远。

whats worse，高维空间中距离容易集中，最近点和最远点的距离差异相对变小，导致 nearest neighbor 的判断依据变弱。
